# Lab 3.4 · Blameless review and regression test

Day 3 · Session 4 (Learning from incidents) · 25 min.

Write a blameless review of a supplied incident and turn its failing cases into a regression test that fails on the bad version and passes on the fixed one.

**Tasks**
- [ ] Read the incident record · 4 min
- [ ] Timeline and contributing factors · 7 min
- [ ] Owned corrective actions · 5 min
- [ ] The regression test: fails before, passes after · 9 min
- [ ] ✱ Stretch · Add the test to the canary gate of Lab 3.3

AI policy: AI-tutor. An assistant may comment on your wording; the contributing factors and the actions are your findings from the record.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ae-403.git"
    if not Path("/content/ae-403").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ae-403"], check=True)
    os.chdir("/content/ae-403/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("_tools").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())
import relia as U
import json
import re
import pandas as pd

## 1 · Read the incident record

`data/incident/INC-0412.json` is the record of the incident you drilled in Lab 3.2: a timeline
in minutes from the page, chat extracts and measured facts. `data/incident/cases.jsonl` holds
twelve failing cases: Arabic tracking questions with the parcel number in Eastern Arabic
digits, and the answers of v2.4 and of the fixed v2.4.1.

In [ ]:
inc = json.loads((U.DATA / "incident" / "INC-0412.json").read_text(encoding="utf-8"))
cases = [json.loads(l) for l in (U.DATA / "incident" / "cases.jsonl").read_text(encoding="utf-8").splitlines()]
print(inc["summary"])
print(pd.DataFrame(inc["timeline"]).to_string(index=False))
print(inc["facts"])

## 2 · Timeline and contributing factors

✏️ **Exercise** · Write the review's timeline (detect, declare, mitigate, assess, notify, with
minutes) and at least four contributing factors. Each factor names a condition of the system
or the process, with the evidence from the record. `blameless(text)` flags blaming language:
names of people, "human error", "careless", "should have", "failed to".

In [ ]:
BLAME = re.compile(r"human error|careless|should have|failed to|negligen|fault of|" +
                   "|".join(re.escape(n) for n in ["Thabit", "Ghufran", "Shuaib", "Sundus", "Ithar", "Ruwaida"]), re.I)


def blameless(text: str) -> bool:
    return BLAME.search(text) is None


def review() -> dict:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    R = review()
except NotImplementedError:
    R = U.reference("lab_3_4_review")
U.keep("lab_3_4_review", R)
print(pd.DataFrame(R["contributing_factors"]).to_string(index=False))
text = json.dumps(R, ensure_ascii=False)
U.ok(blameless(text), "the review names conditions, not people")
U.ok(len(R["contributing_factors"]) >= 4, "at least four contributing factors, each with evidence")

## 3 · Owned corrective actions

✏️ **Exercise** · One action per contributing factor: what changes, the owning role (not a
person), the due date in days, and how you will know it is done (a test, a gate, a drill).

In [ ]:
def actions() -> list[dict]:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    A = actions()
except NotImplementedError:
    A = U.reference("lab_3_4_actions")
U.keep("lab_3_4_actions", A)
print(pd.DataFrame(A).to_string(index=False))
U.ok(all(a["owner"] and a["due_days"] and a["verified_by"] for a in A), "every action has an owner, a due date and a check")
U.ok(blameless(json.dumps(A, ensure_ascii=False)), "owners are roles, not names")

## 4 · The regression test: fails before, passes after

✏️ **Exercise** · Write `check_case(answer, case)` returning a list of problems: the answer
must name the parcel the customer asked about (compare after `U.normalise_digits`), and must
name no other parcel number (`U.PARCEL_RE`). Then run it on every case for both versions.

In [ ]:
def check_case(answer: str, case: dict) -> list[str]:
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    results = {v: sum(1 for c in cases if not check_case(c["answers"][v], c)) for v in ("v2.4", "v2.4.1")}
except NotImplementedError:
    results = U.reference("lab_3_4_results")
U.keep("lab_3_4_results", results)
print("cases passing:", results, "of", len(cases))
U.ok(results == {"v2.4": 0, "v2.4.1": 12}, "the test fails on every case for v2.4 and passes on every case for v2.4.1")

**Look at** where the test lives: a regression test nobody runs is a note. In the capstone,
show it in the release gate, failing on the old version in the pipeline log.

✱ **Stretch** · Add the test to the canary gate of Lab 3.3 as a pre-release step.

## Check yourself

1. Why is "the engineer did not test Arabic" not a contributing factor?
   <details><summary>Answer</summary>It blames a person for a gap in the process: the factor is that the evaluation set had no Arabic tracking questions, which any engineer would have missed.</details>
2. What makes a corrective action complete?
   <details><summary>Answer</summary>An owning role, a due date and a check that proves it is done, such as a test in the release gate.</details>
3. Why must the regression test fail on v2.4?
   <details><summary>Answer</summary>A test that passes on the broken version cannot detect the regression; failing first proves it tests the right thing.</details>

**Carry forward** · The review and the test are the evidence LO4 asks for; the capstone asks for the same on the reviewed service.